# Step 1: Import Required Packages

In [1]:
import os
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_chroma import Chroma
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage

C:\Users\Rohit singh\AppData\Local\Temp\ipykernel_9380\1987215824.py:4: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


# Step 2: Load Gemini API Key

In [2]:
import os

from dotenv import load_dotenv
from langchain_google_genai import ChatGoogleGenerativeAI

load_dotenv()

api_key = os.getenv("GEMINI_API_KEY")

if not api_key:
    print("Gemini API key not found ❌")
else:
    llm = ChatGoogleGenerativeAI(
        model="gemini-3.6-flash",
        temperature=0.7
    )

    print("Gemini LLM initialized successfully")

Gemini LLM initialized successfully


# Step 3: Build Basic Chatbot

In [3]:
while True:
    user_input = input("You: ")

    if user_input.lower() == "exit":
        print("AI: Goodbye! 👋")
        break

    response = llm.invoke(user_input)

    if isinstance(response.content, list):
        ai_text = "\n".join(
            item["text"]
            for item in response.content
            if isinstance(item, dict) and item.get("type") == "text"
        )
    else:
        ai_text = response.content

    print("AI:", ai_text)

You:  exit


AI: Goodbye! 👋


# Step 4: Conversation Memory

In [4]:
chat_history = []

while True:
    user_input = input("You: ")

    if user_input.lower() == "exit":
        print("AI: Goodbye! 👋")
        break

    chat_history.append(
        HumanMessage(content=user_input)
    )

    response = llm.invoke(chat_history)

    if isinstance(response.content, list):
        ai_text = "\n".join(
            item["text"]
            for item in response.content
            if isinstance(item, dict) and item.get("type") == "text"
        )
    else:
        ai_text = response.content

    print("AI:", ai_text)

    chat_history.append(
        AIMessage(content=ai_text)
    )

You:  hello


C:\Users\Rohit singh\AppData\Local\Programs\Python\Python312\Lib\site-packages\langchain_google_genai\chat_models.py:3908: UserWarning: Model 'gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(
Direct use of automatic function calling (AFC) in Models.generate_content is not recommended. Instead, we recommend to use AFC in Chat.send_message. Similarly, direct use of AFC in Models.generate_content_stream is not recommended. Instead, we recommend to use AFC in Chat.send_message_stream.


AI: Hello! How can I help you today?


You:  how are you


C:\Users\Rohit singh\AppData\Local\Programs\Python\Python312\Lib\site-packages\langchain_google_genai\chat_models.py:3908: UserWarning: Model 'gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


AI: I'm doing well, thank you for asking! How are you doing today? How can I help you?


You:  exit


AI: Goodbye! 👋


# Step 5: Add System Prompt

In [6]:
system_prompt = """
You are a helpful and friendly AI assistant.
Answer questions clearly and accurately.
If the user asks something you do not know, say that you are not sure.
Keep your answers easy to understand.
"""

response = llm.invoke([
    SystemMessage(content=system_prompt),
    HumanMessage(content="What is Artificial Intelligence?")
])

if isinstance(response.content, list):
    ai_text = "\n".join(
        item["text"]
        for item in response.content
        if isinstance(item, dict) and item.get("type") == "text"
    )
else:
    ai_text = response.content

C:\Users\Rohit singh\AppData\Local\Programs\Python\Python312\Lib\site-packages\langchain_google_genai\chat_models.py:3908: UserWarning: Model 'gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


# Step 6: System Prompt

In [7]:
system_prompt = """
You are a helpful and friendly AI assistant.

Your job is to:
- Answer questions clearly.
- Explain difficult topics in simple language.
- Give accurate and useful answers.
- If you don't know something, honestly say that you don't know.
- Be polite and professional.
"""

response = llm.invoke([
    SystemMessage(content=system_prompt),
    HumanMessage(content="What is Artificial Intelligence?")
])

if isinstance(response.content, list):
    ai_text = "\n".join(
        item["text"]
        for item in response.content
        if isinstance(item, dict) and item.get("type") == "text"
    )
else:
    ai_text = response.content

print("AI:", ai_text)

C:\Users\Rohit singh\AppData\Local\Programs\Python\Python312\Lib\site-packages\langchain_google_genai\chat_models.py:3908: UserWarning: Model 'gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


AI: **Artificial Intelligence (AI)** is a branch of computer science that creates machines and software capable of thinking, learning, and solving problems—tasks that usually require human intelligence.

Instead of just following strict, pre-written instructions, AI systems analyze large amounts of data to find patterns, make decisions, and improve over time.

---

### Everyday Examples of AI
You likely interact with AI every day without even realizing it:

* **Voice Assistants:** Siri, Alexa, and Google Assistant use AI to understand your spoken language and answer your questions.
* **Recommendation Systems:** Netflix, YouTube, and Spotify use AI to learn what you like and suggest new movies, videos, or songs.
* **Navigation Apps:** Google Maps and Waze use AI to analyze live traffic data and find the fastest route for you.
* **Smart Search Engines:** Google uses AI to understand what you are looking for and give you the most relevant results.
* **Generative AI:** Tools like ChatGPT (

# Step 7: Combine System Prompt with Conversation Memory

In [8]:
system_prompt = """
You are a helpful and friendly AI assistant.

Your job is to:
- Answer questions clearly.
- Explain difficult topics in simple language.
- Give accurate and useful answers.
- Use the previous conversation when answering questions.
- If you don't know something, honestly say that you don't know.
- Be polite and professional.
"""

chat_history = [
    SystemMessage(content=system_prompt)
]

while True:
    user_input = input("You: ")

    if user_input.lower() == "exit":
        print("AI: Goodbye! 👋")
        break

    chat_history.append(
        HumanMessage(content=user_input)
    )

    response = llm.invoke(chat_history)

    if isinstance(response.content, list):
        ai_text = "\n".join(
            item["text"]
            for item in response.content
            if isinstance(item, dict) and item.get("type") == "text"
        )
    else:
        ai_text = response.content

    print("AI:", ai_text)

    chat_history.append(
        AIMessage(content=ai_text)
    )

You:   My name is Rohit.


C:\Users\Rohit singh\AppData\Local\Programs\Python\Python312\Lib\site-packages\langchain_google_genai\chat_models.py:3908: UserWarning: Model 'gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


AI: Hello Rohit! It's very nice to meet you. How can I help you today?


You:  I am learning Python.


C:\Users\Rohit singh\AppData\Local\Programs\Python\Python312\Lib\site-packages\langchain_google_genai\chat_models.py:3908: UserWarning: Model 'gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


...

You:  ok


C:\Users\Rohit singh\AppData\Local\Programs\Python\Python312\Lib\site-packages\langchain_google_genai\chat_models.py:3908: UserWarning: Model 'gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


AI: Great! Whenever you're ready, what Python topic would you like to explore next? We could talk about basic concepts, try out a simple coding example, or answer any questions you have.


You:  exit


AI: Goodbye! 👋


# Step 8: Create Chatbot Function

In [10]:
system_prompt = """
You are a helpful and friendly AI assistant.

Your job is to:
- Answer questions clearly.
- Explain difficult topics in simple language.
- Give accurate and useful answers.
- Use the previous conversation when answering questions.
- If you don't know something, honestly say that you don't know.
- Be polite and professional.
"""


def get_ai_response(chat_history):
    response = llm.invoke(chat_history)

    if isinstance(response.content, list):
        ai_text = "\n".join(
            item["text"]
            for item in response.content
            if isinstance(item, dict) and item.get("type") == "text"
        )
    else:
        ai_text = response.content

    return ai_text


def start_chat():
    chat_history = [
        SystemMessage(content=system_prompt)
    ]

    while True:
        user_input = input("You: ")

        if user_input.lower() == "exit":
            print("AI: Goodbye! 👋")
            break

        chat_history.append(
            HumanMessage(content=user_input)
        )

        ai_text = get_ai_response(chat_history)

        print("AI:", ai_text)

        chat_history.append(
            AIMessage(content=ai_text)


        )


start_chat()

You:  exit


AI: Goodbye! 👋
